In [14]:
#!git clone https://github.com/dramirezbe/DataBase-RF-FM-88MHz-108MHz-Bogota-Funza.git

fatal: destination path 'DataBase-RF-FM-88MHz-108MHz-Bogota-Funza' already exists and is not an empty directory.


In [2]:
import pandas as pd
import glob
from tqdm import tqdm
import numpy as np

archivos = glob.glob("DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node*.csv")
datos_nodos = {}

# ── Load all files ──────────────────────────────────────────────────────────
for archivo in tqdm(archivos, desc="Cargando CSVs"):
    nombre_nodo = archivo.replace('.csv', '')
    datos_nodos[nombre_nodo] = pd.read_csv(archivo)

# ── Consistency Check ───────────────────────────────────────────────────────
print("\n" + "="*60)
print("DATA CONSISTENCY REPORT")
print("="*60)

issues_summary = []

for nodo, df in datos_nodos.items():
    issues = []

    # 1. Empty file (no rows at all)
    if df.empty:
        issues.append("⛔ EMPTY FILE — no rows loaded")

    # 2. Completely empty columns (all NaN)
    empty_cols = df.columns[df.isna().all()].tolist()
    if empty_cols:
        issues.append(f"🟥 Fully-NaN columns: {empty_cols}")

    # 3. NaN counts per column (partial)
    nan_counts = df.isna().sum()
    partial_nan_cols = nan_counts[(nan_counts > 0) & (nan_counts < len(df))]
    if not partial_nan_cols.empty:
        details = ", ".join(f"{col}={n}" for col, n in partial_nan_cols.items())
        issues.append(f"🟧 Partial NaNs — {details}")

    # 4. Fully duplicated rows
    n_dupes = df.duplicated().sum()
    if n_dupes > 0:
        issues.append(f"🟨 Duplicate rows: {n_dupes}")

    # 5. Columns with a single unique value (no variance — likely corrupt/constant)
    constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
    if constant_cols:
        issues.append(f"🟦 Constant/no-variance columns: {constant_cols}")

    # 6. Expected shape sanity check (flag if row count is an outlier vs others)
    row_counts = {k: len(v) for k, v in datos_nodos.items()}
    median_rows = np.median(list(row_counts.values()))
    if len(df) < 0.5 * median_rows:
        issues.append(f"🟪 Suspiciously few rows: {len(df)} (median across nodes: {median_rows:.0f})")

    # ── Print per-node summary ───────────────────────────────────────────────
    status = "✅ OK" if not issues else "⚠️  ISSUES FOUND"
    print(f"\n[{status}] {nodo}  |  shape: {df.shape}")
    for issue in issues:
        print(f"       {issue}")

    if issues:
        issues_summary.append({"node": nodo, "issues": issues})

# ── Global summary ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print(f"SUMMARY: {len(datos_nodos)} nodes checked | "
      f"{len(issues_summary)} with issues | "
      f"{len(datos_nodos) - len(issues_summary)} clean")
print("="*60)

# Optional: export issues to CSV for later review
if issues_summary:
    pd.DataFrame([
        {"node": r["node"], "issue": i}
        for r in issues_summary
        for i in r["issues"]
    ]).to_csv("consistency_report.csv", index=False)
    print("📄 Issues saved to consistency_report.csv")


DATA CONSISTENCY REPORT

[⚠️  ISSUES FOUND] DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node3-Bogota  |  shape: (105, 16)
       🟥 Fully-NaN columns: ['lat', 'lng', 'excursion_peak_to_peak_hz', 'excursion_peak_deviation_hz', 'excursion_rms_deviation_hz', 'depth_peak_to_peak', 'depth_peak_deviation', 'depth_rms_deviation']
       🟦 Constant/no-variance columns: ['mac', 'campaign_id', 'start_freq_hz', 'end_freq_hz', 'lat', 'lng', 'excursion_peak_to_peak_hz', 'excursion_peak_deviation_hz', 'excursion_rms_deviation_hz', 'depth_peak_to_peak', 'depth_peak_deviation', 'depth_rms_deviation']

[⚠️  ISSUES FOUND] DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node8-Bogota  |  shape: (23, 16)
       🟥 Fully-NaN columns: ['lat', 'lng', 'excursion_peak_to_peak_hz', 'excursion_peak_deviation_hz', 'excursion_rms_deviation_hz', 'depth_peak_to_peak', 'depth_peak_deviation', 'depth_rms_deviation']
       🟦 Constant/no-variance columns: ['mac', 'campaign_id', 'start_freq_hz', 'end_freq_hz', 'lat', 'lng', 'excurs

Cargando CSVs: 100%|██████████| 10/10 [00:11<00:00,  1.19s/it]


In [4]:
import pandas as pd
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import ast
import sys

archivos = glob.glob("DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node*.csv")
datos_nodos = {}

# ── Load all files ──────────────────────────────────────────────────────────
EXCLUDE = {
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node8-Bogota",
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node3-Bogota",
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node9-Funza",
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node6-Bogota",
}

for archivo in tqdm(archivos, desc="Loading CSV data"):
    nombre_nodo = archivo.replace('.csv', '')
    if nombre_nodo in EXCLUDE:
        print(f"⏭️  Skipped: {nombre_nodo}")
        continue
    datos_nodos[nombre_nodo] = pd.read_csv(archivo)

# =============================================================================
# 🔍 DATA SHAPE & SIZE DIAGNOSTICS
# =============================================================================
print(f"\n📊 Dataset Summary Report")
print("=" * 70)

# ── 1. Per-Node Shape Information ─────────────────────────────────────────
print(f"\n📋 Per-Node DataFrame Shapes:")
print(f"{'Node Name':<50} {'Rows':>8} {'Cols':>6} {'Memory (MB)':>12}")
print("-" * 70)

node_shapes = {}
total_rows = 0
total_memory_bytes = 0

for nombre_nodo, df in sorted(datos_nodos.items()):
    rows, cols = df.shape
    memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    
    node_shapes[nombre_nodo] = {'rows': rows, 'cols': cols}
    total_rows += rows
    total_memory_bytes += df.memory_usage(deep=True).sum()
    
    display_name = nombre_nodo.split('/')[-1] if '/' in nombre_nodo else nombre_nodo
    print(f"{display_name:<50} {rows:>8} {cols:>6} {memory_mb:>12.2f}")

# ── 2. Aggregate Statistics ───────────────────────────────────────────────
print(f"\n📈 Aggregate Statistics:")
print(f"   • Total nodes loaded:     {len(datos_nodos)}")
print(f"   • Total records (rows):   {total_rows:,}")
print(f"   • Total memory usage:     {total_memory_bytes / (1024**2):.2f} MB")
print(f"   • Average rows per node:  {total_rows / len(datos_nodos):.1f} ± {np.std([s['rows'] for s in node_shapes.values()]):.1f}")

# ── 3. Column Consistency Check ───────────────────────────────────────────
print(f"\n🔍 Column Consistency:")
all_columns = [set(df.columns) for df in datos_nodos.values()]
if len(set(frozenset(c) for c in all_columns)) == 1:
    # Use next(iter()) for first value
    first_df = next(iter(datos_nodos.values()))
    print(f"   ✅ All nodes have identical columns: {list(first_df.columns)}")
else:
    print(f"   ⚠️  Column mismatch detected!")
    for nombre_nodo, df in datos_nodos.items():
        print(f"      {nombre_nodo}: {list(df.columns)}")

# ── 4. Data Type & Sample Preview ─────────────────────────────────────────
if datos_nodos:
    # 🔹 FIX: Use next(iter()) instead of values()[0]
    first_node = next(iter(datos_nodos.values()))
    
    print(f"\n🧪 Sample Data Structure (first node, first row):")
    print(f"   Columns: {list(first_node.columns)}")
    print(f"   Data types:\n{first_node.dtypes}")
    
    if 'pxx' in first_node.columns:
        sample_pxx = first_node['pxx'].iloc[0]
        pxx_array = np.array(ast.literal_eval(sample_pxx))
        print(f"\n   'pxx' sample:")
        print(f"      • Raw string length: {len(sample_pxx)} chars")
        print(f"      • Parsed array shape: {pxx_array.shape}")
        print(f"      • Value range: [{pxx_array.min():.2f}, {pxx_array.max():.2f}]")
        print(f"      • First 5 values: {pxx_array[:5]}")

# ── 5. Row Index Range Validation ─────────────────────────────────────────
print(f"\n🔢 Row Index Ranges (for data_record selection):")
index_ranges = {}
for nombre_nodo, df in datos_nodos.items():
    min_idx, max_idx = df.index.min(), df.index.max()
    index_ranges[nombre_nodo] = (min_idx, max_idx)
    display_name = nombre_nodo.split('/')[-1]
    print(f"   {display_name:<30} [{min_idx}, {max_idx}]")

# Global safe range for data_record
global_min = max(r[0] for r in index_ranges.values())
global_max = min(r[1] for r in index_ranges.values())
print(f"\n   ✅ Safe data_record range for ALL nodes: [{global_min}, {global_max}]")

# ── 6. Optional: Visual Summary ───────────────────────────────────────────
if len(datos_nodos) > 1:
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    node_labels = [n.split('/')[-1] for n in node_shapes.keys()]
    row_counts = [s['rows'] for s in node_shapes.values()]
    plt.barh(node_labels, row_counts, color='steelblue', alpha=0.8)
    plt.xlabel("Number of Records")
    plt.title("Records per Node")
    plt.grid(axis='x', alpha=0.3)
    
    plt.subplot(1, 2, 2)
    memory_vals = [df.memory_usage(deep=True).sum() / (1024**2) for df in datos_nodos.values()]
    plt.barh(node_labels, memory_vals, color='seagreen', alpha=0.8)
    plt.xlabel("Memory (MB)")
    plt.title("Memory Usage per Node")
    plt.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ── 7. Export Summary for Reproducibility ─────────────────────────────────
data_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'n_nodes': len(datos_nodos),
    'total_records': total_rows,
    'total_memory_mb': total_memory_bytes / (1024**2),
    'node_shapes': node_shapes,
    'safe_row_range': (global_min, global_max),
    'columns': list(first_node.columns) if datos_nodos else [],
    'excluded_nodes': list(EXCLUDE)
}

print(f"\n💾 Summary dictionary stored in 'data_summary'")
print(f"   Example access: data_summary['safe_row_range'] → {data_summary['safe_row_range']}")

⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node3-Bogota
⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node8-Bogota
⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node6-Bogota
⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node9-Funza

📊 Dataset Summary Report

📋 Per-Node DataFrame Shapes:
Node Name                                              Rows   Cols  Memory (MB)
----------------------------------------------------------------------
Node1-Bogota                                            105     16        63.87
Node10-Bogota                                           104     16        63.24
Node2-Bogota                                            105     16        63.82
Node4-Bogota                                            105     16        63.85
Node5-Bogota                                            105     16        63.86
Node7-Bogota                                            105     16        63.85

📈 Aggregate Statistics:
   • Total nodes load

Loading CSV data: 100%|██████████| 10/10 [00:07<00:00,  1.36it/s]


In [16]:
import pandas as pd
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import ast

archivos = glob.glob("DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node*.csv")
datos_nodos = {}

# ── Load all files ──────────────────────────────────────────────────────────
EXCLUDE = {
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node8-Bogota", # not enough data
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node3-Bogota", # poorest measure quality
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node9-Funza",  # poor quality
    "DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node6-Bogota", # poor quality 
}

for archivo in tqdm(archivos, desc="Loading  CSV data"):
    nombre_nodo = archivo.replace('.csv', '')
    if nombre_nodo in EXCLUDE:
        print(f"⏭️  Skipped: {nombre_nodo}")
        continue
    datos_nodos[nombre_nodo] = pd.read_csv(archivo)


from scipy.stats import pearsonr

global_max = min(r[1] for r in index_ranges.values()) + 1

# =============================================================================
# LOOP: data_record from 0 to global_max 
# =============================================================================
all_ranked_scores   = {}   # {node_name: [score_record_0, score_record_1, ...]}
all_cumulative_data = []   # list of (data_record, ranked_nodes, ranked_scores)

for data_record in tqdm(range(global_max), desc="Processing records"):

    pxx_indexed           = {}
    noise_floor_estimates = {}

    # ── PASS 1: Noise floor + recentered PSDs ────────────────────────────────
    for nombre_nodo, df in datos_nodos.items():
        pxx_raw = df['pxx'].iloc[data_record]
        pxx     = np.array(ast.literal_eval(pxx_raw))
        pxx_indexed[nombre_nodo] = pxx

        counts, bins      = np.histogram(pxx, bins=50)
        noise_floor_db    = bins[np.argmax(counts)]
        noise_floor_estimates[nombre_nodo] = noise_floor_db

    noise_array       = np.array(list(noise_floor_estimates.values()))
    global_noise_mean = np.mean(noise_array)

    pxx_processed = {}
    for nombre_nodo in datos_nodos.keys():
        pxx_orig      = pxx_indexed[nombre_nodo]
        offset        = global_noise_mean - noise_floor_estimates[nombre_nodo]
        pxx_recentered = pxx_orig + offset
        pxx_norm      = (pxx_recentered - np.mean(pxx_recentered)) / (np.std(pxx_recentered) + 1e-8)
        pxx_processed[nombre_nodo] = pxx_norm

    # ── PASS 2: Pairwise Pearson correlation matrix ───────────────────────────
    node_names  = sorted(datos_nodos.keys())
    n_nodes     = len(node_names)
    corr_matrix = np.zeros((n_nodes, n_nodes))

    for i, node_i in enumerate(node_names):
        for j, node_j in enumerate(node_names):
            if i == j:
                corr_matrix[i, j] = 1.0
            elif j > i:
                corr_val, _        = pearsonr(pxx_processed[node_i], pxx_processed[node_j])
                corr_matrix[i, j]  = corr_val
                corr_matrix[j, i]  = corr_val

    # ── PASS 3: Cumulative scores + ranking ───────────────────────────────────
    cumulative_scores = np.sum(corr_matrix, axis=1) - 1.0
    ranking_idx       = np.argsort(cumulative_scores)[::-1]
    ranked_nodes      = [node_names[i] for i in ranking_idx]
    ranked_scores     = cumulative_scores[ranking_idx]

    all_cumulative_data.append((data_record, ranked_nodes, ranked_scores))

    # Accumulate per-node scores for averaging
    for node, score in zip(node_names, cumulative_scores):
        all_ranked_scores.setdefault(node, []).append(score)

# =============================================================================
# AVERAGE cumulative correlation score per node across all records
# =============================================================================
avg_scores   = {node: np.mean(scores) for node, scores in all_ranked_scores.items()}
avg_sorted   = dict(sorted(avg_scores.items(), key=lambda x: x[1], reverse=True))

print("\n📊 Average Cumulative Correlation Score per Node")
print("-" * 60)
print(f"{'Rank':<6} {'Node':<35} {'Avg Score':>10} {'Avg Corr':>10}")
print("-" * 60)
for rank, (node, avg) in enumerate(avg_sorted.items(), 1):
    print(f"{rank:<6} {node:<35} {avg:>10.4f} {avg/(n_nodes-1):>10.4f}")

# =============================================================================
# PLOT: Cumulative correlation scores per node across all data_records
# =============================================================================
short_name = lambda n: n.split('/')[-1]   # strip path prefix for readability

cmap    = plt.cm.get_cmap('tab20', n_nodes)
records = list(range(global_max))

# =============================================================================
# PLOT: Cumulative correlation scores per node — raw trajectories only
# =============================================================================
short_name = lambda n: n.split('/')[-1]

# 🔹 Determine actual number of processed records from data
n_processed = len(next(iter(all_ranked_scores.values())))
records = list(range(n_processed))

print(f"📈 Plotting {n_processed} records × {len(datos_nodos)} nodes...")

# Use a colormap with enough distinct colors
cmap = plt.cm.get_cmap('tab20', len(datos_nodos))

fig, ax = plt.subplots(figsize=(18, 9))

# ── Plot each node's score trajectory ──────────────────────────────────────
for idx, node in enumerate(sorted(datos_nodos.keys())):
    if node not in all_ranked_scores:
        continue
        
    scores_over_time = all_ranked_scores[node]
    
    # 🔹 Safety: ensure lengths match
    if len(scores_over_time) != len(records):
        min_len = min(len(scores_over_time), len(records))
        scores_over_time = scores_over_time[:min_len]
        plot_records = records[:min_len]
    else:
        plot_records = records
    
    # Plot line + markers at each record
    ax.plot(plot_records, scores_over_time,
            label=short_name(node),
            color=cmap(idx),
            linewidth=1.8,
            marker='o',
            markersize=5,
            markeredgecolor='white',
            markeredgewidth=0.5,
            alpha=0.9)

# ── Axis labels and title ──────────────────────────────────────────────────
ax.set_xlabel("Data Record Index", fontsize=13, fontweight='medium')
ax.set_ylabel("Cumulative Correlation Score", fontsize=13, fontweight='medium')
ax.set_title("Cumulative Pearson Correlation Score per Node\n"
             "Markers show score at each individual record",
             fontsize=14, fontweight='bold', pad=15)

# ── X-axis: adaptive tick spacing ──────────────────────────────────────────
if n_processed <= 30:
    ax.set_xticks(records)
else:
    step = max(1, n_processed // 12)
    ax.set_xticks(records[::step])
    ax.set_xticklabels([str(r) for r in records[::step]], rotation=45, ha='right')

ax.set_xlim(-0.5, n_processed - 0.5)
ax.set_axisbelow(True)

# ── Legend: outside plot to avoid clutter ──────────────────────────────────
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1),
          fontsize=8.5, title="Nodes", title_fontsize=10,
          borderaxespad=0, frameon=True, fancybox=True)

# ── Grid and layout ────────────────────────────────────────────────────────
ax.grid(True, which='major', linestyle=':', linewidth=0.5, alpha=0.4)
ax.grid(True, which='minor', linestyle=':', linewidth=0.3, alpha=0.2)
plt.tight_layout()
plt.show()
# =============================================================================
# SUMMARY TABLE: Average scores ranked
# =============================================================================
summary_df = pd.DataFrame({
    'Node':       list(avg_sorted.keys()),
    'Avg_Score':  list(avg_sorted.values()),
    'Avg_Corr':   [v / (n_nodes - 1) for v in avg_sorted.values()]
}).reset_index(drop=True)
summary_df.index += 1
summary_df.index.name = 'Rank'
print("\n", summary_df.to_string())

⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node3-Bogota
⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node8-Bogota
⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node6-Bogota
⏭️  Skipped: DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node9-Funza

📊 Average Cumulative Correlation Score per Node
------------------------------------------------------------
Rank   Node                                 Avg Score   Avg Corr
------------------------------------------------------------
1      DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node2-Bogota     4.5077     0.9015
2      DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node5-Bogota     4.4848     0.8970
3      DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node1-Bogota     4.4808     0.8962
4      DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node10-Bogota     4.4727     0.8945
5      DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node7-Bogota     4.4653     0.8931
6      DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node4-Bogota     4.4610     0.892

Processing records: 100%|██████████| 104/104 [03:27<00:00,  1.99s/it]
/tmp/ipykernel_324/1611752184.py:107: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap    = plt.cm.get_cmap('tab20', n_nodes)
/tmp/ipykernel_324/1611752184.py:122: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', len(datos_nodos))


In [17]:
# language: python
import ast
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

DATA_DIR = Path("DataBase-RF-FM-88MHz-108MHz-Bogota-Funza")
files = sorted(DATA_DIR.glob("Node*.csv"))

# Normalize EXCLUDE to basenames (without suffix)
EXCLUDE = {
    "Node8-Bogota",
    "Node3-Bogota",
    "Node9-Funza",
    "Node6-Bogota",
}

# Load CSVs and parse pxx column once
datos_nodos = {}
load_errors = {}

for f in files:
    base = f.stem  # basename without .csv
    if base in EXCLUDE:
        print(f"⏭️  Skipped: {base}")
        continue
    try:
        df = pd.read_csv(f)
    except Exception as e:
        load_errors[base] = f"read error: {e}"
        continue
    if 'pxx' not in df.columns:
        load_errors[base] = "missing 'pxx' column"
        continue

    # Parse pxx column to list of numpy arrays; keep track of rows that fail
    parsed = []
    parse_ok = True
    for i, raw in enumerate(df['pxx'].astype(str)):
        try:
            arr = np.array(ast.literal_eval(raw), dtype=float)
        except Exception as e:
            parse_ok = False
            load_errors.setdefault(base, []).append((i, f"parse error: {e}"))
            break
        parsed.append(arr)

    if not parse_ok:
        continue

    # Check all arrays same length
    lengths = {a.size for a in parsed}
    if len(lengths) != 1:
        load_errors[base] = f"Inconsistent pxx length(s): {sorted(lengths)}"
        continue

    df = df.copy()
    df['_pxx_arr'] = parsed
    datos_nodos[base] = df

if not datos_nodos:
    raise RuntimeError("No valid nodes loaded. errors: " + repr(load_errors))

# Determine number of records to process: minimum rows among nodes
min_rows = min(len(df) for df in datos_nodos.values())
print(f"Processing {min_rows} records across {len(datos_nodos)} nodes")

node_names = sorted(datos_nodos.keys())
n_nodes = len(node_names)
pxx_len = next(iter(datos_nodos.values()))['_pxx_arr'][0].size

all_ranked_scores = defaultdict(list)
all_cumulative_data = []

for rec_idx in tqdm(range(min_rows), desc="Processing records"):
    # Collect pxx arrays for this record
    pxx_list = []
    for name in node_names:
        arr = datos_nodos[name]['_pxx_arr'][rec_idx]
        pxx_list.append(arr)
    # Convert to 2D array (n_nodes, pxx_len)
    data = np.vstack(pxx_list)  # shape (n_nodes, pxx_len)

    # Noise-floor estimate per-node using histogram bin centers
    noise_floor = []
    for row in data:
        counts, bins = np.histogram(row, bins=50)
        bin_centers = (bins[:-1] + bins[1:]) / 2.0
        noise_floor.append(bin_centers[np.argmax(counts)])
    noise_floor = np.array(noise_floor)
    global_noise_mean = noise_floor.mean()

    # Recenter and normalize rows
    recentered = data + (global_noise_mean - noise_floor)[:, None]
    means = recentered.mean(axis=1, keepdims=True)
    stds = recentered.std(axis=1, keepdims=True)
    # Avoid division by zero: mark constant rows
    const_mask = (stds.flatten() < 1e-12)
    safe_stds = np.where(stds < 1e-12, 1.0, stds)
    normed = (recentered - means) / safe_stds

    # If a row is constant (std zero), set its normalized values to zero to avoid NaNs
    normed[const_mask, :] = 0.0

    # Fast correlation matrix
    corr_matrix = np.corrcoef(normed)
    # Replace NaNs (can occur if some rows were all zeros after normalization) with 0
    corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)

    # Cumulative score: sum of correlations to other nodes
    cumulative_scores = corr_matrix.sum(axis=1) - 1.0  # subtract self-corr
    ranking_idx = np.argsort(cumulative_scores)[::-1]
    ranked_nodes = [node_names[i] for i in ranking_idx]
    ranked_scores = cumulative_scores[ranking_idx]

    all_cumulative_data.append((rec_idx, ranked_nodes, ranked_scores))
    for name, score in zip(node_names, cumulative_scores):
        all_ranked_scores[name].append(score)

# Average scores
avg_scores = {n: np.mean(s) for n, s in all_ranked_scores.items()}
avg_sorted = dict(sorted(avg_scores.items(), key=lambda x: x[1], reverse=True))

print("\n📊 Average Cumulative Correlation Score per Node")
print("-" * 60)
print(f"{'Rank':<6} {'Node':<25} {'Avg Score':>12} {'Avg Corr':>12}")
print("-" * 60)
for rank, (node, avg) in enumerate(avg_sorted.items(), 1):
    print(f"{rank:<6} {node:<25} {avg:>12.4f} {avg/(n_nodes-1):>12.4f}")

# Plot
n_processed = min_rows
records = np.arange(n_processed)

cmap = plt.cm.get_cmap('hsv', n_nodes)
fig, ax = plt.subplots(figsize=(14, 8))
for idx, node in enumerate(node_names):
    scores = all_ranked_scores[node]
    ax.plot(records, scores, label=node, color=cmap(idx), linewidth=1.5, alpha=0.9)

ax.set_xlabel("Data Record Index")
ax.set_ylabel("Cumulative Correlation Score")
ax.set_title("Cumulative Pearson Correlation Score per Node")
ax.grid(True, linestyle=':', alpha=0.4)
# Place legend to the right and adjust layout
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout(rect=[0, 0, 0.85, 1.0])
plt.show()

⏭️  Skipped: Node3-Bogota
⏭️  Skipped: Node6-Bogota
⏭️  Skipped: Node8-Bogota
⏭️  Skipped: Node9-Funza
Processing 104 records across 6 nodes

📊 Average Cumulative Correlation Score per Node
------------------------------------------------------------
Rank   Node                         Avg Score     Avg Corr
------------------------------------------------------------
1      Node2-Bogota                    4.5077       0.9015
2      Node5-Bogota                    4.4848       0.8970
3      Node1-Bogota                    4.4808       0.8962
4      Node10-Bogota                   4.4727       0.8945
5      Node7-Bogota                    4.4653       0.8931
6      Node4-Bogota                    4.4610       0.8922


Processing records: 100%|██████████| 104/104 [00:00<00:00, 152.39it/s]
/tmp/ipykernel_324/2403885766.py:138: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('hsv', n_nodes)


In [18]:
# language: python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Parameters: adjust as needed
ROLL_WINDOW = 9            # rolling window size in records (odd is nice)
MIN_PERIODS = 3            # minimum non-NaN observations in window
ZSCORE_THRESH = 3.0        # threshold for rolling z-score detection
ROBUST_MULT = 3.0          # multiplier for MAD-based detection

# Build DataFrame of scores: rows = records, columns = node short names
node_order = sorted(all_ranked_scores.keys())
# n_processed determined earlier (use length of any node series)
n_processed = len(next(iter(all_ranked_scores.values())))
records = np.arange(n_processed)

scores_df = pd.DataFrame({n: pd.Series(all_ranked_scores[n], index=records)
                          for n in node_order})

# Rolling mean/std (sensitive) and rolling median/MAD (robust)
rolling_mean = scores_df.rolling(window=ROLL_WINDOW, min_periods=MIN_PERIODS, center=True).mean()
rolling_std  = scores_df.rolling(window=ROLL_WINDOW, min_periods=MIN_PERIODS, center=True).std(ddof=0)

rolling_median = scores_df.rolling(window=ROLL_WINDOW, min_periods=MIN_PERIODS, center=True).median()
# MAD: median(|x - median|)
rolling_mad = (scores_df - rolling_median).abs().rolling(window=ROLL_WINDOW, min_periods=MIN_PERIODS, center=True).median()

# Avoid division by zero: replace very small std/MAD with tiny epsilon
EPS = 1e-12
rolling_std_safe = rolling_std.replace(0, EPS).fillna(EPS)
rolling_mad_safe = rolling_mad.replace(0, EPS).fillna(EPS)

# Z-score based deviation (sensitive)
z_scores = (scores_df - rolling_mean) / rolling_std_safe

# Robust score: deviation divided by MAD (scaled to be comparable to std)
# For normal distributions, std ≈ 1.4826 * MAD, so scale MAD accordingly
MAD_TO_STD = 1.4826
robust_scores = (scores_df - rolling_median).abs() / (rolling_mad_safe * MAD_TO_STD)

# Flags: True where either method exceeds thresholds
z_flags     = (z_scores.abs() >= ZSCORE_THRESH)
robust_flags= (robust_scores >= ROBUST_MULT)
combined_flags = z_flags | robust_flags

# Summarize detected events
events = []
for node in node_order:
    flagged_idx = combined_flags.index[combined_flags[node]].tolist()
    for idx in flagged_idx:
        events.append({
            'record': int(idx),
            'node': node,
            'score': float(scores_df.at[idx, node]),
            'z_score': float(z_scores.at[idx, node]) if not pd.isna(z_scores.at[idx, node]) else np.nan,
            'robust_score': float(robust_scores.at[idx, node]) if not pd.isna(robust_scores.at[idx, node]) else np.nan,
            'z_flag': bool(z_flags.at[idx, node]) if not pd.isna(z_flags.at[idx, node]) else False,
            'robust_flag': bool(robust_flags.at[idx, node]) if not pd.isna(robust_flags.at[idx, node]) else False
        })

events_df = pd.DataFrame(events)
events_df = events_df.sort_values(['record', 'node']).reset_index(drop=True)

print(f"\n🔔 Detected {len(events_df)} transient candidate events (combined criteria).")
if not events_df.empty:
    print(events_df.head(30).to_string(index=False))

# Optional: per-node transient counts
transient_counts = events_df.groupby('node').size().rename('n_transients').sort_values(ascending=False)
print("\nTransient counts per node:")
print(transient_counts.to_string())

# Optional: Plot scores and overlay flags for a subset of nodes (or all)
PLOT_NODES = node_order  # or node_order[:8] to limit
cmap = plt.cm.get_cmap('tab20', len(PLOT_NODES))

fig, ax = plt.subplots(figsize=(14, 8))
for i, node in enumerate(PLOT_NODES):
    ax.plot(records, scores_df[node], label=node, color=cmap(i), linewidth=1.5, alpha=0.9)
    # Mark z-score flags with small markers
    flagged = combined_flags[node]
    if flagged.any():
        ax.scatter(records[flagged], scores_df[node][flagged],
                   color='black', s=40, marker='x', zorder=5, label=f"{node} transient" if i==0 else None)

ax.set_xlabel("Data Record Index")
ax.set_ylabel("Cumulative Correlation Score")
ax.set_title("Cumulative Scores with Detected Transient Events")
ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout(rect=[0, 0, 0.85, 1.0])
plt.show()


🔔 Detected 20 transient candidate events (combined criteria).
 record          node    score   z_score  robust_score  z_flag  robust_flag
      1  Node4-Bogota 4.413818 -1.940872  4.378875e+00   False         True
      9  Node4-Bogota 4.400940 -1.474288  3.304828e+00   False         True
     12  Node2-Bogota 4.274547 -2.362861  4.497085e+00   False         True
     38  Node2-Bogota 4.351300 -2.153210  3.965956e+00   False         True
     38  Node7-Bogota 4.361386 -1.666138  3.021339e+00   False         True
     41  Node2-Bogota 4.492239  1.067167  5.195929e+00   False         True
     46  Node2-Bogota 4.412462 -1.580613  3.383972e+00   False         True
     51  Node5-Bogota 4.531061  1.301327  2.334737e+10   False         True
     53  Node5-Bogota 4.496446 -1.060945  1.145192e+01   False         True
     60  Node1-Bogota 4.552729  1.688547  4.707087e+00   False         True
     61  Node1-Bogota 4.467557 -1.347872  3.371040e+00   False         True
     67  Node7-Bogota 4.4

/tmp/ipykernel_324/694101372.py:76: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', len(PLOT_NODES))


In [19]:
# language: python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----------------------
# EWM (Exponentially Weighted) parameters
# ----------------------
EWM_SPAN = 20           # larger -> smoother / slower response
EWM_MIN_PERIODS = 3
EWM_ZSCORE_THRESH = 3.0  # threshold (z-score) for EWM detection

# ----------------------
# Compute EWM mean and variance, then std
# ----------------------
# Use center of mass via span to control smoothing:
ewm_mean = scores_df.ewm(span=EWM_SPAN, min_periods=EWM_MIN_PERIODS, adjust=False).mean()
# ewm var: use .var(ddof=0) on ewm (pandas has ewm().var())
ewm_var = scores_df.ewm(span=EWM_SPAN, min_periods=EWM_MIN_PERIODS, adjust=False).var(bias=True)
ewm_std = np.sqrt(ewm_var).replace(0, 1e-12).fillna(1e-12)

# ----------------------
# EWM z-scores and flags
# ----------------------
ewm_z = (scores_df - ewm_mean) / ewm_std
ewm_flags = ewm_z.abs() >= EWM_ZSCORE_THRESH

# ----------------------
# Combine all methods: rolling z, robust (MAD), EWM
# (Assumes z_scores, robust_flags, combined_flags from previous block exist)
# ----------------------
# Already-computed:
# z_flags         = (z_scores.abs() >= ZSCORE_THRESH)
# robust_flags    = (robust_scores >= ROBUST_MULT)
# combined_flags  = z_flags | robust_flags

# Create a DataFrame consolidating method flags
method_flags = pd.concat({
    'rolling_z': z_flags,
    'robust_mad': robust_flags,
    'ewm_z': ewm_flags
}, axis=1)
# axis shape: MultiIndex columns (method, node); easier to compute per-method counts:
method_flags_counts = {
    'rolling_z': z_flags.sum().sum(),
    'robust_mad': robust_flags.sum().sum(),
    'ewm_z': ewm_flags.sum().sum()
}
print("\nMethod total flag counts:", method_flags_counts)

# ----------------------
# Merge contiguous flagged records into events per method & node
# ----------------------
def merge_flags_to_events(flag_series):
    """Given a boolean Series indexed by record, return list of (start, end) inclusive event tuples."""
    events = []
    if flag_series.empty:
        return events
    is_flagged = flag_series.fillna(False).astype(bool)
    # Find contiguous True regions
    diff = is_flagged.astype(int).diff().fillna(is_flagged.iloc[0].astype(int))
    starts = is_flagged.index[(diff == 1)].tolist()
    ends   = is_flagged.index[(diff == -1)].tolist()
    # Edge cases: starts/ends alignment
    if is_flagged.iloc[0]:
        if not starts or (starts and starts[0] != is_flagged.index[0]):
            starts = [is_flagged.index[0]] + starts
    if is_flagged.iloc[-1]:
        if not ends or (ends and ends[-1] != is_flagged.index[-1]):
            ends = ends + [is_flagged.index[-1]]
    # Pair them
    for s, e in zip(starts, ends):
        events.append((int(s), int(e)))
    return events

# Build a summary of event counts (per method/node) and durations
event_summaries = []
methods = ['rolling_z', 'robust_mad', 'ewm_z']
for method in methods:
    flags_df = {'rolling_z': z_flags, 'robust_mad': robust_flags, 'ewm_z': ewm_flags}[method]
    for node in flags_df.columns:
        node_flags = flags_df[node]
        node_events = merge_flags_to_events(node_flags)
        for (s, e) in node_events:
            event_summaries.append({
                'method': method,
                'node': node,
                'start': s,
                'end': e,
                'duration': e - s + 1
            })

events_summary_df = pd.DataFrame(event_summaries)
if events_summary_df.empty:
    print("\nNo merged events detected by any method.")
else:
    # Quick aggregate counts and mean duration per method
    agg = events_summary_df.groupby('method').agg(
        n_events=('node', 'size'),
        mean_duration=('duration', 'mean'),
        median_duration=('duration', 'median')
    ).reset_index()
    print("\nEvent aggregates by method:")
    print(agg.to_string(index=False))
    # Top nodes by number of events per method
    print("\nTop nodes with events (per method):")
    top_nodes = events_summary_df.groupby(['method', 'node']).size().reset_index(name='n_events')
    print(top_nodes.sort_values(['method', 'n_events'], ascending=[True, False]).groupby('method').head(5).to_string(index=False))

# ----------------------
# Optional: Export events_summary_df and events_df (previous combined) to CSV
# ----------------------
# events_summary_df.to_csv("detected_events_summary.csv", index=False)
# events_df.to_csv("detected_instances_raw.csv", index=False)

# ----------------------
# Plot comparison for a selected node (or nodes) showing raw score, rolling median+MAD, EWM mean, and flags
# ----------------------
PLOT_NODES = node_order[:6]  # change slice to view more/fewer nodes
fig, axes = plt.subplots(len(PLOT_NODES), 1, figsize=(14, 3 * len(PLOT_NODES)), sharex=True)
if len(PLOT_NODES) == 1:
    axes = [axes]

for ax, node in zip(axes, PLOT_NODES):
    ax.plot(records, scores_df[node], label='score', color='C0', lw=1.5)
    ax.plot(records, rolling_mean[node], label='rolling_mean', color='C1', lw=1.2, alpha=0.8)
    ax.plot(records, rolling_median[node], label='rolling_median', color='C2', lw=1.2, alpha=0.8, linestyle='--')
    ax.plot(records, ewm_mean[node], label=f'EWM_mean (span={EWM_SPAN})', color='C3', lw=1.2, alpha=0.9)

    # Mark flags for each method
    ax.scatter(records[z_flags[node]], scores_df[node][z_flags[node]], marker='x', color='magenta', label='rolling_z_flag', s=30)
    ax.scatter(records[robust_flags[node]], scores_df[node][robust_flags[node]], marker='o', facecolors='none', edgecolors='green', label='robust_mad_flag', s=40)
    ax.scatter(records[ewm_flags[node]], scores_df[node][ewm_flags[node]], marker='D', color='black', label='ewm_z_flag', s=30)

    ax.set_ylabel(f"{node}")
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    ax.grid(alpha=0.3, linestyle=':')

axes[-1].set_xlabel("Data Record Index")
plt.tight_layout(rect=[0, 0, 0.85, 1.0])
plt.show()

# ----------------------
# Quick numeric comparison summary: overlap between methods
# ----------------------
# Fraction of flags overlap between methods pairwise
pairs = [('rolling_z', 'ewm_z'), ('rolling_z', 'robust_mad'), ('robust_mad', 'ewm_z')]
overlap_results = []
for a, b in pairs:
    A = {'rolling_z': z_flags, 'robust_mad': robust_flags, 'ewm_z': ewm_flags}[a]
    B = {'rolling_z': z_flags, 'robust_mad': robust_flags, 'ewm_z': ewm_flags}[b]
    # Flatten boolean arrays
    total_flags_a = A.values.sum()
    total_flags_b = B.values.sum()
    both = (A & B).values.sum()
    union = (A | B).values.sum()
    overlap_results.append({
        'pair': f"{a} vs {b}",
        'a_flags': int(total_flags_a),
        'b_flags': int(total_flags_b),
        'both': int(both),
        'union': int(union),
        'jaccard': (both / union) if union > 0 else np.nan,
        'pct_a_in_b': (both / total_flags_a) if total_flags_a > 0 else np.nan,
        'pct_b_in_a': (both / total_flags_b) if total_flags_b > 0 else np.nan
    })

print("\nMethod-pair overlap (Jaccard and containment):")
print(pd.DataFrame(overlap_results).to_string(index=False))


Method total flag counts: {'rolling_z': 0, 'robust_mad': 20, 'ewm_z': 0}

Event aggregates by method:
    method  n_events  mean_duration  median_duration
robust_mad        19            2.0              2.0

Top nodes with events (per method):
    method          node  n_events
robust_mad  Node2-Bogota         5
robust_mad  Node4-Bogota         4
robust_mad Node10-Bogota         3
robust_mad  Node5-Bogota         3
robust_mad  Node1-Bogota         2

Method-pair overlap (Jaccard and containment):
                   pair  a_flags  b_flags  both  union  jaccard  pct_a_in_b  pct_b_in_a
     rolling_z vs ewm_z        0        0     0      0      NaN         NaN         NaN
rolling_z vs robust_mad        0       20     0     20      0.0         NaN         0.0
    robust_mad vs ewm_z       20        0     0     20      0.0         0.0         NaN
